In [ ]:
import requests

carrinho = []


def incluir_item(carrinho, nome, preco, quantidade):
    if preco <= 0 or quantidade <= 0:
        print(f"Erro: '{nome}' não foi adicionado (preço e quantidade devem ser maiores que zero).")
        return

    item = {"nome": nome, "preco": preco, "quantidade": quantidade}
    carrinho.append(item)
    print(f"'{nome}' adicionado ao carrinho.")


def buscarItem(carrinho, nomeBuscado):
    for item in carrinho:
        if item["nome"].lower() == nomeBuscado.lower():
            print(f"\nItem encontrado: {item['nome']}")
            print(f"Preço: R$ {item['preco']:.2f}")
            print(f"Quantidade: {item['quantidade']}")
            return item

    print(f"\nItem '{nomeBuscado}' não encontrado no carrinho.")
    return None


def calcular_subtotal(carrinho):
    subtotal = 0
    for item in carrinho:
        subtotal += item["preco"] * item["quantidade"]
    return subtotal


def calcular_desconto(subtotal, vip=False):
    if subtotal >= 500:
        taxa = 0.10
    elif subtotal >= 200:
        taxa = 0.05
    else:
        taxa = 0.0

    if vip:
        taxa += 0.05

    taxa = min(taxa, 0.15)
    return taxa


def consultar_estado_por_cep(cep):
    cep_limpo = ''.join(filter(str.isdigit, cep))
    if len(cep_limpo) != 8:
        raise ValueError('CEP inválido. Deve conter exatamente 8 dígitos.')

    url = f'https://viacep.com.br/ws/{cep_limpo}/json/'
    response = requests.get(url, timeout=5)

    if response.status_code == 200:
        dados = response.json()
        if 'erro' in dados:
            raise ValueError('CEP não encontrado na base de dados.')
        return dados.get('uf')
    else:
        raise Exception('Erro de conexão com o serviço ViaCEP.')


def obter_frete_por_estado(cep):
    uf = consultar_estado_por_cep(cep)

    tabela_frete = {
        'SP': 15.00,
        'RJ': 20.00,
        'MG': 22.00,
        'ES': 25.00,
        'PR': 28.00,
        'SC': 28.00,
        'RS': 32.00,
        'BA': 35.00,
        'DF': 35.00,
        'AM': 50.00,
    }

    valor_frete = tabela_frete.get(uf, 40.00)
    return {'estado': uf, 'valor_frete': round(valor_frete, 2)}


def aplicar_cupom(valor, cupom=None):
    if cupom == "PRIMEIRACOMPRA":
        return valor * 0.97
    return valor


def exibir_resumo(carrinho, subtotal, taxa_desconto, valor_desconto, frete, estado, total):
    print("\n--- Resumo do Pedido ---")

    if len(carrinho) == 0:
        print("O carrinho está vazio.")
        return

    for item in carrinho:
        total_item = item["preco"] * item["quantidade"]
        print(f"{item['nome']} | R$ {item['preco']:.2f} x {item['quantidade']} = R$ {total_item:.2f}")

    print(f"\nSubtotal: R$ {subtotal:.2f}")
    print(f"Desconto: {taxa_desconto * 100:.0f}% (R$ {valor_desconto:.2f})")
    print(f"Frete ({estado}): R$ {frete:.2f}")
    print(f"Total final: R$ {total:.2f}")


saudacao = """========================================
        Bem-vindo(a) à VarejoSmart!
========================================"""
print(saudacao)

cliente = input("Insira seu nome: \n\n")

incluir_item(carrinho, "Livro", 50.00, 1)
incluir_item(carrinho, "Caneta", 10.00, 1)
incluir_item(carrinho, "Item inválido", 0, 2)

buscarItem(carrinho, "Livro")

subtotal = calcular_subtotal(carrinho)
taxa_desconto = calcular_desconto(subtotal, vip=False)
valor_desconto = subtotal * taxa_desconto
valor_com_desconto = subtotal - valor_desconto

cepCliente = input("\nInsira seu CEP para calcular o frete: ").strip()

try:
    resultado_frete = obter_frete_por_estado(cepCliente)
    frete = resultado_frete['valor_frete']
    estado = resultado_frete['estado']
except (ValueError, Exception) as erro:
    print(f"Aviso: {erro} Frete padrão de R$40,00 será aplicado.")
    frete = 40.00
    estado = "N/A"

total = valor_com_desconto + frete
total = aplicar_cupom(total, cupom=None)

exibir_resumo(carrinho, subtotal, taxa_desconto, valor_desconto, frete, estado, total)

Insira seu nome: 

Henrique

Insira o nome do item que você deseja adicionar ao carrinho: jaca´re

Insira o preco deste item: 13.3

Insira a quantidade deste item: 3

Insira o nome do item que deseja buscar: jaca´re

Item encontrado: jaca´re
Preço: R$ 13.30
Quantidade: 3


In [ ]:
"""try:
    cep_destino = '20040-002'  # Exemplo de CEP do Rio de Janeiro
    resultado = obter_frete_por_estado(cep_destino)

    print(f"Estado de Destino: {resultado['estado']}")
    print(f"Valor do Frete Fixo: R$ {resultado['valor_frete']:.2f}")

    except Exception as error:
    print(f'Falha no cálculo: {error}')"""